# 03 — Gold Feature Engineering

Build the **driver-race feature mart** from Silver Parquet using **PySpark** + **DuckDB** validation.

- **Engine:** `GOLD_ENGINE = "spark"`. Spark is the official engine; pandas fallback is manual only (`allow_fallback=True`).
- **Grain:** one row per `session_key`, `meeting_key`, `driver_number`
- **Base:** `session_result_clean.parquet`
- **Target:** `points_finish` = 1 if `points` > 0, else 0
- **No modeling** in this notebook — features and leakage guard only

Run after `02_silver_cleaning_quality.ipynb` with the same `USE_GOOGLE_DRIVE` setting.


## Connection to grading rubric

| Rubric area | This notebook |
|-------------|----------------|
| Gold layer | Feature mart at `data/gold/driver_race_feature_mart.parquet` |
| Target variable | `points_finish` from session points |
| Feature engineering | Lap, pit, position, weather, race control, metadata |
| Data quality | Gold DQ reports under `reports/data_quality/` |
| Leakage prevention | `gold_leakage_guard_report.csv` + feature dictionary |


## Colab setup (required every session)

Identical to `00`–`02`: clone, `pip install -e .`, Drive mount, set `OPENF1_DATA_ROOT` **before** importing config.


In [ ]:
import os
import subprocess
import sys
from pathlib import Path

# A. Repository (code)
REPO_URL = "https://github.com/dk546/openf1-big-data-pipeline.git"
REPO_NAME = "openf1-big-data-pipeline"
PROJECT_ROOT = Path(f"/content/{REPO_NAME}")

# B. Google Drive persistence (outputs) — set OPENF1_DATA_ROOT before importing config
USE_GOOGLE_DRIVE = True
DRIVE_OUTPUT_ROOT = Path("/content/drive/MyDrive/openf1_big_data_pipeline")

if USE_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE_OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
    os.environ["OPENF1_DATA_ROOT"] = str(DRIVE_OUTPUT_ROOT)
    print("OPENF1_DATA_ROOT:", os.environ["OPENF1_DATA_ROOT"])
else:
    os.environ.pop("OPENF1_DATA_ROOT", None)
    print("OPENF1_DATA_ROOT not set — repo-local outputs.")

# D. Clone or update repository
if not PROJECT_ROOT.exists():
    print("Cloning repository...")
    subprocess.run(["git", "clone", REPO_URL, str(PROJECT_ROOT)], check=True)
else:
    print("Updating repository...")
    subprocess.run(["git", "-C", str(PROJECT_ROOT), "pull"], check=False)

os.chdir(PROJECT_ROOT)
print("Working directory:", Path.cwd())

# E. Verify project files
_checks = {
    "README.md": PROJECT_ROOT / "README.md",
    "pyproject.toml": PROJECT_ROOT / "pyproject.toml",
    "src/openf1_pipeline": PROJECT_ROOT / "src" / "openf1_pipeline",
    "src/openf1_pipeline/__init__.py": PROJECT_ROOT / "src" / "openf1_pipeline" / "__init__.py",
    "src/openf1_pipeline/config.py": PROJECT_ROOT / "src" / "openf1_pipeline" / "config.py",
}
for name, path in _checks.items():
    if not path.exists():
        raise FileNotFoundError(f"Missing {name}: {path}")

# F. Install dependencies and editable package
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", "."], check=True)

# G. Fallback: src on sys.path
_src = PROJECT_ROOT / "src"
if str(_src) not in sys.path:
    sys.path.insert(0, str(_src))

# H. Import test
import openf1_pipeline  # noqa: E402

from openf1_pipeline.config import (  # noqa: E402
    ensure_project_directories,
    get_artifacts_dir,
    get_bronze_dir,
    get_data_dir,
    get_gold_dir,
    get_output_root,
    get_project_root,
    get_reports_dir,
    get_silver_dir,
)

paths = ensure_project_directories()

# I. Path summary
print("PROJECT_ROOT:", get_project_root())
print("OUTPUT_ROOT:", get_output_root())
print("DATA_DIR:", get_data_dir())
print("BRONZE_DIR:", get_bronze_dir())
print("SILVER_DIR:", get_silver_dir())
print("GOLD_DIR:", get_gold_dir())
print("REPORTS_DIR:", get_reports_dir())
print("ARTIFACTS_DIR:", get_artifacts_dir())



## Start Spark


In [ ]:
from openf1_pipeline.utils.spark import get_spark

spark = get_spark()
print("Spark version:", spark.version)


## Silver prerequisites


In [ ]:
from pathlib import Path

import pandas as pd

from openf1_pipeline.config import (
    get_data_quality_reports_dir,
    get_feature_definitions_dir,
    get_gold_dir,
    get_output_root,
    get_silver_dir,
)
from openf1_pipeline.gold.build_feature_mart import build_gold_feature_mart
from openf1_pipeline.utils.io import read_parquet_if_exists
from openf1_pipeline.utils.cleanup import clean_gold_layer_outputs

GOLD_ENGINE = "spark"
ALLOW_FALLBACK = False
CLEAR_GOLD_OUTPUTS = True

SILVER_DIR = get_silver_dir()
GOLD_DIR = get_gold_dir()
DATA_QUALITY_REPORTS_DIR = get_data_quality_reports_dir()
FEATURE_DEFINITIONS_DIR = get_feature_definitions_dir()

print("OUTPUT_ROOT:", get_output_root())
print("SILVER_DIR:", SILVER_DIR)
print("GOLD_DIR:", GOLD_DIR)

required = [
    SILVER_DIR / "session_result_clean.parquet",
    SILVER_DIR / "laps_clean.parquet",
    SILVER_DIR / "drivers_clean.parquet",
]
for path in required:
    if not path.exists():
        raise FileNotFoundError(
            f"Missing Silver table: {path}. Run 02_silver_cleaning_quality.ipynb first."
        )

starting_grid = SILVER_DIR / "starting_grid_clean.parquet"
if starting_grid.exists():
    sg = read_parquet_if_exists(starting_grid)
    if sg is not None and sg.empty:
        print("WARNING: starting_grid_clean.parquet is empty — grid features skipped.")
else:
    print("NOTE: starting_grid_clean.parquet not found (optional).")


## Clean Gold outputs (required before fresh Spark run)

Removes Gold Parquet, Gold DQ reports, DuckDB Gold reports, and feature dictionary.
Does not delete Silver.


In [ ]:
if CLEAR_GOLD_OUTPUTS:
    print("Cleaning Gold layer outputs...")
    clean_gold_layer_outputs(
        gold_dir=GOLD_DIR,
        data_quality_reports_dir=DATA_QUALITY_REPORTS_DIR,
        feature_definitions_dir=FEATURE_DEFINITIONS_DIR,
    )
else:
    print("Skipping Gold cleanup (CLEAR_GOLD_OUTPUTS=False).")


## Silver pre-flight: defensive join-key dtype check

Before building Gold, confirm that join keys (`session_key`, `meeting_key`,
`driver_number`, `lap_number`, `position`) are stored as integer/long on
disk in every Silver Parquet. If a key is floating, joins still work in
Spark — but downstream DuckDB validation and per-key counts are clearer
when types are integral.

`build_race_control_features` intentionally ignores `race_control.qualifying_phase`
because that column is 100% null in current OpenF1 pulls (see the Silver
data-quality notes). Gold features use `message` and `flag` only.

If this cell prints a warning, rerun Notebook 02 with the latest
`clean_*_spark` modules (they now cast keys to long explicitly) and then
restart this notebook from the Silver pre-flight cell.

In [ ]:
from openf1_pipeline.utils.spark import spark_path

KEY_COLUMNS_PER_TABLE = {
    "meetings": ["meeting_key", "year"],
    "sessions": ["session_key", "meeting_key", "year"],
    "drivers": ["session_key", "driver_number", "meeting_key"],
    "laps": ["session_key", "driver_number", "lap_number"],
    "pit": ["session_key", "driver_number", "lap_number"],
    "weather": ["session_key"],
    "position": ["session_key", "driver_number", "meeting_key", "position"],
    "race_control": ["session_key", "meeting_key"],
    "session_result": ["session_key", "driver_number", "meeting_key", "position"],
    "starting_grid": ["session_key", "driver_number", "position"],
}
INTEGRAL_DTYPES = {"int", "bigint", "smallint", "tinyint", "long"}

silver_schema_rows = []
for table_name, key_cols in KEY_COLUMNS_PER_TABLE.items():
    parquet_path = SILVER_DIR / f"{table_name}_clean.parquet"
    if not parquet_path.exists():
        silver_schema_rows.append({
            "table": table_name, "column": "(missing)", "dtype": "n/a", "is_integer": False,
        })
        continue
    sdf = spark.read.parquet(spark_path(parquet_path))
    schema = dict(sdf.dtypes)
    for col in key_cols:
        if col not in schema:
            continue
        dtype = schema[col]
        silver_schema_rows.append({
            "table": table_name, "column": col, "dtype": dtype,
            "is_integer": dtype.lower() in INTEGRAL_DTYPES,
        })

silver_schema_check = pd.DataFrame(silver_schema_rows)
display(silver_schema_check)

floating_keys = silver_schema_check.loc[
    ~silver_schema_check["is_integer"] & (silver_schema_check["dtype"] != "n/a")
]
if floating_keys.empty:
    print("OK: all Silver join keys are integer/long on disk — safe to build Gold.")
else:
    print("WARNING: the following Silver join keys are floating on disk —")
    display(floating_keys)
    print(
        "Rerun Notebook 02 with the latest clean_*_spark fixes (keys cast to long), "
        "then rerun this cell. Gold will still build, but join-key types should be "
        "integral for consistency with the data dictionary."
    )

## Build Gold feature mart (Spark-first)


In [ ]:
outputs = build_gold_feature_mart(
    silver_dir=SILVER_DIR,
    gold_dir=GOLD_DIR,
    data_quality_reports_dir=DATA_QUALITY_REPORTS_DIR,
    feature_definitions_dir=FEATURE_DEFINITIONS_DIR,
    engine=GOLD_ENGINE,
    spark=spark,
    allow_fallback=ALLOW_FALLBACK,
)

outputs["summary"]


## DuckDB validation (Gold Parquet)


In [ ]:
from openf1_pipeline.analytics.duckdb_validation import (
    save_duckdb_validation_reports,
    validate_gold_with_duckdb,
)

gold_duckdb = validate_gold_with_duckdb(outputs["paths"]["driver_race_feature_mart"])
duckdb_gold_paths = save_duckdb_validation_reports(
    gold_duckdb, DATA_QUALITY_REPORTS_DIR, prefix="gold"
)
duckdb_gold_paths


## Validate target distribution


In [ ]:
target_dist = pd.read_csv(outputs["paths"]["gold_target_distribution"])
display(target_dist)


## Validate feature missingness (top 20)


In [ ]:
missingness = pd.read_csv(outputs["paths"]["gold_feature_missingness"])
display(
    missingness.sort_values("missing_pct", ascending=False).head(20)
)


## Validate join quality


In [ ]:
join_quality = pd.read_csv(outputs["paths"]["gold_join_quality_report"])
display(join_quality)


## Leakage guard


In [ ]:
leakage = pd.read_csv(outputs["paths"]["gold_leakage_guard_report"])
forbidden = leakage[leakage["allowed_for_modeling"] == False]
print(f"Columns blocked from modeling: {len(forbidden)}")
display(leakage[leakage["leakage_risk"].isin(["high", "target"])])
print(f"Model-safe feature count: {len(outputs['model_feature_columns'])}")


## Feature dictionary


In [ ]:
feature_dict = pd.read_csv(outputs["paths"]["feature_dictionary"])
display(feature_dict.head(20))
print("Predictive features:")
display(
    feature_dict[feature_dict["modeling_role"] == "predictive_feature"][
        ["feature_name", "feature_group", "missing_pct"]
    ].head(25)
)


## Inspect Gold output


In [ ]:
gold_df = pd.read_parquet(outputs["paths"]["driver_race_feature_mart"])
print("Shape:", gold_df.shape)
print("Grain keys:", ["session_key", "meeting_key", "driver_number"])
display(gold_df.head(10))
print("Target rate (points_finish=1):", gold_df["points_finish"].mean())


## Checklist / next steps

- [ ] Copy Gold parquet + DQ CSVs to `evidence/<run_id>/` after smoke/full Colab run
- [ ] Confirm `gold_leakage_guard_report.csv` has no forbidden columns with `allowed_for_modeling=True`
- [ ] Proceed to **04 — modeling**
- [ ] Do **not** use `final_position`, `result_*`, or raw `position`/`points` as model inputs
